In [41]:
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix


In [42]:
# right click to upload transformer.py and dataset.py to colab runtime

import sys
sys.path.append('/content')

from transformer import TransformerCls
from custom_dataset import ReviewDataset
from utils import report_metrices

In [43]:
from google.colab import drive
drive.mount("/content/drive")

proj_path = '/content/drive/MyDrive/CSE 3521/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [65]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(proj_path + "modern/custom_tokenizer.json")

In [45]:
training_set = ReviewDataset(proj_path + "data/train.csv")
val_set = ReviewDataset(proj_path + "data/val.csv")
test_set = ReviewDataset(proj_path + "data/test.csv")

training_loader = DataLoader(training_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [46]:
# sqrt-inverse class weights, normalised to mean 1
class_counts = torch.tensor(
    training_set.data["Sentiment"].value_counts().sort_index().tolist(),
    dtype=torch.float,
)

class_weights = 1.0 / torch.sqrt(class_counts)
class_weights = class_weights / class_weights.mean()

print("class counts :", class_counts.tolist())
print("class weights:", [round(w, 4) for w in class_weights.tolist()])


class counts : [4783.0, 7022.0, 20613.0]
class weights: [1.3004, 1.0732, 0.6264]


In [79]:
model_config = {
    "embedding_dim": 64,
    "nhead": 8,
    "num_layers": 3,
    "dropout": 0.3
}

In [80]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerCls(tokenizer, **model_config) 
model = model.to(device)

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=0.1,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

In [63]:
def train_one_epoch():
    total_loss = 0
    total_samples = 0

    for i, data in enumerate(training_loader):

        inputs, labels = data
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        if i % 500 == 0:
            print(f"batch {i} loss: {total_loss / total_samples}")
    return total_loss / total_samples

In [64]:
n_epoch = 25
patience = 3

best_f1 = -1.0
best_epoch = -1
bad_epochs = 0

for epoch in range(n_epoch):
    print(f"for epoch {epoch}")

    model.train(True)
    avg_loss = train_one_epoch()

    model.eval()
    y_val_true = []
    y_val_pred = []
    with torch.no_grad():
        for vinputs, vlabels in val_loader:
            voutputs = model(vinputs)
            y_val_pred.append(voutputs.argmax(dim=1).cpu())
            y_val_true.append(vlabels)

    y_val_true = torch.cat(y_val_true).numpy()
    y_val_pred = torch.cat(y_val_pred).numpy()
    metrices = report_metrices(y_val_true, y_val_pred)
    val_f1 = metrices["f1"]["macro"]

    scheduler.step(val_f1)

    if val_f1 > best_f1 + 1e-6:
        best_f1 = val_f1
        best_epoch = epoch
        bad_epochs = 0
        torch.save(model.state_dict(), proj_path + "modern/experiment/transformer.pth")
        print(f"  saved (best val macro-F1 {best_f1:.4f})")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"early stop: no val macro-F1 gain for {bad_epochs} epochs")
            break

print(f"best epoch {best_epoch}  val macro-F1 {best_f1:.4f}")

for epoch 0
batch 0 loss: 1.1318504810333252
batch 500 loss: 1.0503103877018074
batch 1000 loss: 1.0007908401432095
F1         per-class [0.48127490039840637, 0.4040978941377348, 0.8288547664747281]  macro 0.571  micro 0.701  weighted 0.686
Precision  per-class [0.527972027972028, 0.4714475431606906, 0.7834895675839129]  macro 0.594  micro 0.701  weighted 0.678
Recall     per-class [0.44216691068814057, 0.35358565737051795, 0.8797962648556876]  macro 0.559  micro 0.701  weighted 0.701
Accuracy   0.701
Confusion Matrix  labels [0, 1, 2]
[[ 302  146  235]
 [ 168  355  481]
 [ 102  252 2591]]
  saved (best val macro-F1 0.5714)
for epoch 1
batch 0 loss: 0.8193747401237488
batch 500 loss: 0.8909054111815736
batch 1000 loss: 0.881092609701814
F1         per-class [0.5966057441253264, 0.4593562465902891, 0.8374300728937109]  macro 0.631  micro 0.723  weighted 0.720
Precision  per-class [0.5382803297997645, 0.5078407720144753, 0.8361543669600542]  macro 0.627  micro 0.723  weighted 0.721
Recal

In [81]:
model.load_state_dict(torch.load(proj_path + "modern/transformer.pth", weights_only=True))
model.eval()

y_true = []
y_pred = []
with torch.no_grad():
    for text, label in test_loader:
        output = model(text)
        y_pred.append(output.argmax(dim=1).cpu())
        y_true.append(label)

y_true = torch.cat(y_true).numpy()
y_pred = torch.cat(y_pred).numpy()
metrices = report_metrices(y_true, y_pred)

F1         per-class [0.724813100747597, 0.5336842105263158, 0.8691784845179156]  macro 0.709  micro 0.778  weighted 0.775
Precision  per-class [0.7059639389736477, 0.5652173913043478, 0.8592998174879708]  macro 0.710  micro 0.778  weighted 0.773
Recall     per-class [0.7446964155084126, 0.5054835493519442, 0.8792869269949066]  macro 0.710  micro 0.778  weighted 0.778
Accuracy   0.778
Confusion Matrix  labels [0, 1, 2]
[[1018  218  131]
 [ 275 1014  717]
 [ 149  562 5179]]


In [82]:
import json
with open(proj_path + 'modern/transformer_test_result.json', 'w') as f:
    json.dump(metrices, f)

with open(proj_path + "modern/transformer_config.json", 'w') as f:
    json.dump(model_config, f)